# Pipeline NLP e Análise TF-IDF: Selena Gomez

Este notebook implementa o pipeline ponta a ponta para coleta, pré-processamento e modelagem vetorial (TF-IDF) das músicas da cantora **Selena Gomez**, integrando os conceitos de Web Scraping e Extração de Features em NLP.

## 1. Instalação e Importação de Bibliotecas

Garantimos a presença de todas as dependências essenciais para requisições web, manipulação de dados, expressões regulares e processamento de linguagem natural (`NLTK` e `Spacy`).

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import time
import math
import spacy
import nltk

# Download dos pacotes necessários do NLTK
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

# Carregamento do modelo Spacy para lematização em inglês
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Baixando o modelo en_core_web_sm do Spacy...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

## 2. Coleta de Dados (Web Scraping - Vagalume)

Acessamos a página da artista no Vagalume para capturar a lista alfabética de músicas e, em seguida, iteramos sobre cada link para extrair a letra original e o álbum correspondente com pausas educadas para evitar bloqueios de IP.

In [ ]:
url_artista = "https://www.vagalume.com.br/selena-gomez/"
page = requests.get(url_artista)
soup = BeautifulSoup(page.content, 'html.parser')

# Encontrando a lista alfabética de músicas
lista_alfabetica = BeautifulSoup(str(soup.find_all(id="alfabetMusicList")), 'html.parser')
a_tag = lista_alfabetica.find_all('a')

musicas_lista = []
for a in a_tag:
    nome_musica = a.text
    if not(nome_musica == 'TRADUÇÃO' or nome_musica == ''):
        link_musica = a['href']
        musicas_lista.append([nome_musica, link_musica])

print(f"Total de músicas encontradas na listagem: {len(musicas_lista)}")

# Para demonstração ágil e completa, vamos percorrer e extrair letras e álbuns
print("Iniciando a captura das letras...")
for i in range(len(musicas_lista)):
    link = "https://www.vagalume.com.br" + str(musicas_lista[i][1])
    page_musica = requests.get(link)
    soup_musica = BeautifulSoup(page_musica.content, 'html.parser')
    
    h3_tag = soup_musica.find_all('h3')
    if len(h3_tag) != 0:
        album = h3_tag[0].text
    else:
        album = ''
        
    lyrics = soup_musica.find_all(id='lyrics')
    if len(lyrics) > 0:
        lyrics_str = str(lyrics[0])
        lyrics_str = lyrics_str.replace('<div id="lyrics">', '')
        lyrics_str = lyrics_str.replace('<div data-plugin="googleTranslate" id="lyrics">', '')
        lyrics_str = lyrics_str.replace('<br/>', ' ')
        lyrics_str = lyrics_str.replace("\'", "'")
        lyrics_str = lyrics_str.replace('</div>', '')
    else:
        lyrics_str = ''
        
    musicas_lista[i].append(album)
    musicas_lista[i].append(lyrics_str)
    
    # Pausa entre requisições para evitar sobrecarga no servidor
    time.sleep(0.2)

# Criando o DataFrame inicial
musicas = pd.DataFrame(musicas_lista, columns=['Nome da Música', 'link', 'album', 'letra'])
# Removendo entradas sem letra capturada
musicas = musicas[musicas['letra'].str.strip() != ''].reset_index(drop=True)
print(f"Músicas capturadas com sucesso: {len(musicas)}")
musicas.head()

## 3. Pré-processamento e Limpeza (Língua Inglesa)

Adaptamos a função `limpar_texto` para o idioma inglês: minúsculas, filtragem de caracteres alfabéticos padronizados (`[a-zA-Z]+`), remoção de *stop words* do NLTK e lematização avançada via `Spacy`.

In [ ]:
from nltk.corpus import stopwords

stop_words_english = set(stopwords.words('english'))

def limpar_texto(texto):
    # 1. Minúsculas
    texto_min = texto.lower()
    
    # 2. Selecionar apenas letras do alfabeto inglês com REGEX
    apenas_letras = re.findall(r'[a-zA-Z]+', texto_min)
    texto_junto = " ".join(apenas_letras)
    
    # 3. Lematização e filtragem de stop words
    doc = nlp(texto_junto)
    tokens_limpos = []
    for token in doc:
        # Exclui stop words tanto na forma original quanto lematizada
        if token.text not in stop_words_english and token.lemma_ not in stop_words_english:
            tokens_limpos.append(token.lemma_)
            
    return " ".join(tokens_limpos)

# Atualizando o DataFrame com a nova coluna pré-processada
print("Realizando o pré-processamento das letras...")
musicas['letra_limpa'] = musicas['letra'].apply(limpar_texto)

# Filtrando eventuais registros vazios após a limpeza
musicas = musicas[musicas['letra_limpa'].str.strip() != ''].reset_index(drop=True)
musicas.head()

## 4. Definição do Vocabulário Global

Construímos a lista de ocorrências únicas (*Bag of Words* global) a partir dos tokens de todas as músicas processadas.

In [ ]:
from nltk import word_tokenize

Vocab = []
for letra in musicas['letra_limpa']:
    tokens = word_tokenize(letra)
    for token in tokens:
        if token not in Vocab:
            Vocab.append(token)

print(f"Vocabulário global construído com {len(Vocab)} termos únicos.")

## 5. Modelagem e Cálculo da Métrica TF-IDF

Implementamos as funções manuais para determinar as frequências dos termos (TF), o inverso da frequência nos documentos (IDF) e o escore combinado TF-IDF.

In [ ]:
def dicionario_de_contagem(vocabulario, documento):
    '''Retorna um dicionário com o número de vezes que cada palavra do vocabulário ocorre no documento.'''
    dic = dict.fromkeys(vocabulario, 0)
    for palavra in documento:
        if palavra in dic:
            dic[palavra] = dic[palavra] + 1
    return dic

def calculaTF(dic_de_cont, doc):
    '''Calcula o Term Frequency normalizado para o documento.'''
    tf_dic = {}
    num_palavras_doc = len(doc)
    for palavra, contagem in dic_de_cont.items():
        tf_dic[palavra] = contagem / float(num_palavras_doc)
    return tf_dic

def calculaIDF(lista_de_docs):
    '''Calcula o Inverse Document Frequency global em escala logarítmica na base 10.'''
    idf_dic = {}
    N = len(lista_de_docs)
    for palavra in lista_de_docs[0]:
        num_docs_aparece = 0
        for doc in lista_de_docs:
            if doc[palavra] > 0:
                num_docs_aparece += 1
        idf_dic[palavra] = math.log10(N / float(num_docs_aparece))
    return idf_dic

def calculaTFIDF(tf_bow, idfs):
    '''Multiplica TF pelo IDF correspondente para cada termo.'''
    tfidf = {}
    for palavra in tf_bow:
        tf = tf_bow[palavra]
        idf = idfs[palavra]
        tfidf[palavra] = tf * idf
    return tfidf

### Execução do Fluxo para todas as Músicas

Mapeamos cada música como um documento independente para computar a matriz completa.

In [ ]:
print("1. Gerando contagem de termos por documento...")
lista_dic_cont = []
lista_tokens_docs = []

for letra in musicas['letra_limpa']:
    tokens = word_tokenize(letra)
    lista_tokens_docs.append(tokens)
    dic_cont = dicionario_de_contagem(Vocab, tokens)
    lista_dic_cont.append(dic_cont)

print("2. Calculando TF...")
lista_tf_bow = []
for dic_cont, tokens in zip(lista_dic_cont, lista_tokens_docs):
    tf_bow = calculaTF(dic_cont, tokens)
    lista_tf_bow.append(tf_bow)

print("3. Calculando IDF global...")
idfs = calculaIDF(lista_dic_cont)

print("4. Consolidando pontuações TF-IDF...")
lista_tfidf_docs = []
for tf_bow in lista_tf_bow:
    tfidf = calculaTFIDF(tf_bow, idfs)
    lista_tfidf_docs.append(tfidf)

## 6. Apresentação Final dos Resultados

Organizamos as métricas calculadas em um **DataFrame bem formatado**, onde as colunas representam o Nome da Música e cada termo do vocabulário global.

In [ ]:
# Estruturando a tabela final de pontuações TF-IDF
tfidf_rows = []
for i, tfidf_doc in enumerate(lista_tfidf_docs):
    row = {'Nome da Música': musicas.loc[i, 'Nome da Música']}
    row.update(tfidf_doc)
    tfidf_rows.append(row)

tabela_tfidf = pd.DataFrame(tfidf_rows)

# Exibindo o resultado final formatado
print(f"Dimensões da matriz final: {tabela_tfidf.shape[0]} músicas x {tabela_tfidf.shape[1]-1} termos no vocabulário")
tabela_tfidf.head(10)